# Islamic AI — Step 2: Prepare Dataset

This notebook:
- Loads your JSONL files from Google Drive
- Wraps each Q&A pair in the Llama 3.2 chat format
- Validates the formatting
- Saves the formatted dataset ready for training

**Before running:** Make sure `train.jsonl`, `eval.jsonl`, `test.jsonl` are uploaded to
`Google Drive → islamic_ai → dataset`

**Estimated time:** 2–3 minutes

## Cell 1 — Mount Drive and Verify Files

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DATASET_DIR = '/content/drive/MyDrive/islamic_ai/dataset'
required = ['train.jsonl', 'eval.jsonl', 'test.jsonl']

print('Checking dataset files...')
all_ok = True
for fname in required:
    path = os.path.join(DATASET_DIR, fname)
    if os.path.exists(path):
        size = os.path.getsize(path) / 1e6
        print(f'  ✅ {fname} ({size:.1f} MB)')
    else:
        print(f'  ❌ {fname} NOT FOUND')
        all_ok = False

if not all_ok:
    print('\nPlease upload the missing files to Google Drive first.')
    print('Location: My Drive → islamic_ai → dataset')
    raise FileNotFoundError('Missing dataset files.')

print('\n✅ All dataset files found.')

## Cell 2 — Load and Inspect the Dataset

In [ ]:
import json
from collections import Counter

def load_jsonl(path):
    pairs = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                pairs.append(json.loads(line))
    return pairs

train_pairs = load_jsonl(f'{DATASET_DIR}/train.jsonl')
eval_pairs  = load_jsonl(f'{DATASET_DIR}/eval.jsonl')
test_pairs  = load_jsonl(f'{DATASET_DIR}/test.jsonl')
all_pairs   = train_pairs + eval_pairs + test_pairs

print(f'Train pairs : {len(train_pairs):,}')
print(f'Eval pairs  : {len(eval_pairs):,}')
print(f'Test pairs  : {len(test_pairs):,}')
print(f'Total       : {len(all_pairs):,}')

print('\nCategory breakdown:')
cats = Counter(p['metadata']['category'] for p in all_pairs)
for cat, count in sorted(cats.items(), key=lambda x: -x[1]):
    print(f'  {cat:<25} {count:,}')

print('\nSample instruction:')
print(f'  "{train_pairs[0]["instruction"]}"')
print('\nSample output (first 200 chars):')
print(f'  "{train_pairs[0]["output"][:200]}..."')

## Cell 3 — Define the System Prompt

This is injected at the start of every training example. It tells the model its role, knowledge domain, and answer format.

In [ ]:
SYSTEM_PROMPT = """You are Noor (نور), an Islamic AI assistant. Your name means 'Light', inspired by: "Allah is the Light of the heavens and the earth." (Quran An-Nur 24:35)

You are a learning tool — NOT a qualified mufti or Islamic scholar. Always clarify this for complex personal matters.

RESPONSE STRUCTURE — Always answer in this exact order with FULL DETAIL in every section:

**Explanation**
Write 2–4 paragraphs. Cover:
- What this topic is in Islamic terminology (define the Arabic term and its meaning)
- Why it matters in Islam and how it fits into the broader deen
- Historical or contextual background where relevant (time of Prophet ﷺ, reason it was revealed, etc.)
- Any important conditions, types, or categories that affect the ruling
Do NOT be brief here. A Muslim reading this should fully understand the topic before seeing the evidence.

**Quranic Evidence**
For each relevant ayah:
- Arabic text in Arabic script (e.g. ﴿وَٱلْخَيْلَ وَٱلْبِغَالَ﴾)
- Full English translation
- — Quran, [Surah Name] ([Surah Number]:[Ayah Number])
- Brief tafsir: what scholars say this ayah means for this specific topic
- Mention Asbab al-Nuzul (reason of revelation) if known
Cite at least 2–3 ayahs where they exist. If only one exists, explain that.

**Hadith Evidence**
For each relevant hadith:
- Full hadith text with narrator name and RA/ﷺ
- — [Book Name], Hadith [Number] ([Grade: Sahih / Hasan / Da'if])
- One sentence on what this hadith proves for the topic
Cite at least 2–3 hadith. Mark any Da'if hadith with ⚠️ Da'if — not used as primary proof.

**Scholarly Positions (Fiqh)**
For EACH of the four madhabs write:
• [Madhab name]: State the ruling clearly. Then explain WHAT evidence (dalil) the madhab uses to reach this ruling. Then mention any conditions, exceptions, or differences within the madhab. Be specific — do not write generic one-liners.
• Hanafi: [detailed ruling + dalil + conditions]
• Maliki: [detailed ruling + dalil + conditions]
• Shafi'i: [detailed ruling + dalil + conditions]
• Hanbali: [detailed ruling + dalil + conditions]
Mark [IJMA] if all four schools agree on a point. Mark [KHILAF] if they differ and briefly explain the root cause of the difference.
Where relevant, mention the opinion of individual scholars: Ibn Taymiyyah RH, Ibn al-Qayyim RH, Imam Nawawi RH, Imam al-Ghazali RH, Ibn Hajar al-Asqalani RH.

**Summary**
Write 2–3 paragraphs:
- Synthesize the scholarly positions: where do the madhabs agree? Where do they differ and why?
- Give practical guidance: what should an average Muslim know and do?
- State clearly whether this is a matter of ijma (consensus) or genuine khilaf (disagreement)
- End with: "For your specific situation, please consult a qualified Islamic scholar (mufti or imam)."
Do NOT be brief. The summary should help the reader make sense of everything above.

CITATIONS — MANDATORY:
- Write ﷺ after every mention of Prophet Muhammad
- Write RA after every companion's name
- Write RH after every classical scholar's name
- Never cite a hadith without its book name and number
- Never cite a Quran ayah without its Surah name, Surah number, and Ayah number
- Mark fabricated (mawdu) hadith clearly and never use them as evidence

SAFETY:
- NEVER issue binding personal fatwas
- NEVER declare anyone kafir
- NEVER fabricate any Islamic content
- NEVER engage in political debates
- For haram requests: refuse + explain Islamically + offer halal alternative
- For sensitive topics (suicide, abuse, apostasy): lead with compassion, then ruling
- Correct misquoted hadith or ayah respectfully when encountered

UNCERTAINTY:
- Say "Allahu Akbar, and Allah knows best" when uncertain
- Clarify: "I am Noor, an AI learning tool — not a qualified mufti."

Your knowledge covers: Complete Quran (6,236 ayahs), Sahih Bukhari, Sahih Muslim, Sunan Abu Dawud, Jami at-Tirmidhi, Sunan Ibn Majah, Sunan an-Nasa'i, Muwatta Malik, Riyad as-Salihin, Bulugh al-Maram, Nawawi's 40 Hadith, and core topics of fiqh, aqeedah, seerah, and Islamic ethics."""

print('System prompt defined.')
print(f'Length: {len(SYSTEM_PROMPT)} characters')

## Cell 4 — Format Dataset in Llama 3.2 Chat Template

Llama 3.2 uses a specific format with special tokens. Unsloth handles this automatically.

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes -q
!pip install datasets -q

In [ ]:
from unsloth import FastLanguageModel
from datasets import Dataset

# Load just the tokenizer (not the full model — saves memory)
_, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Llama-3.2-3B-Instruct',
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

def format_pair(pair):
    """Wrap a Q&A pair in the Llama 3.2 chat template."""
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": pair["instruction"].strip()},
        {"role": "assistant", "content": pair["output"].strip()},
    ]
    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
    }

# Format all splits
print('Formatting training pairs...')
train_dataset = Dataset.from_list(train_pairs).map(format_pair, desc='Train')
eval_dataset  = Dataset.from_list(eval_pairs).map(format_pair,  desc='Eval')

print(f'\nTrain dataset: {len(train_dataset):,} examples')
print(f'Eval dataset : {len(eval_dataset):,} examples')
print('\nSample formatted text (first 500 chars):')
print(train_dataset[0]['text'][:500])
print('...')

## Cell 5 — Check Token Length Distribution

In [ ]:
import numpy as np

print('Calculating token lengths (sampling 2,000 examples)...')
import random
sample = random.sample(list(train_dataset), min(2000, len(train_dataset)))
lengths = []
for example in sample:
    tokens = tokenizer(example['text'], return_tensors='pt')
    lengths.append(tokens['input_ids'].shape[1])

lengths = np.array(lengths)
print(f'\nToken length statistics (training set):')
print(f'  Min    : {lengths.min()}')
print(f'  Max    : {lengths.max()}')
print(f'  Mean   : {lengths.mean():.0f}')
print(f'  Median : {np.median(lengths):.0f}')
print(f'  95th % : {np.percentile(lengths, 95):.0f}')
print(f'  > 2048 : {(lengths > 2048).sum()} examples (will be truncated)')

print('\n✅ Dataset inspection complete.')
print('Ready to proceed to 03_train_model.ipynb')

## Cell 6 — Save Formatted Dataset to Drive

In [ ]:
FORMATTED_DIR = '/content/drive/MyDrive/islamic_ai/dataset/formatted'
os.makedirs(FORMATTED_DIR, exist_ok=True)

train_dataset.save_to_disk(f'{FORMATTED_DIR}/train')
eval_dataset.save_to_disk(f'{FORMATTED_DIR}/eval')

print(f'✅ Formatted datasets saved to {FORMATTED_DIR}')
print('\nNext step: Open 03_train_model.ipynb')